# Rewrite Rules Explorer

Interactive notebook for exploring different matrix rewrite rules and visualizing their progressions.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from test_eval import (
    create_matrix_graph, create_functional_node, 
    set_matrix_curve_type, set_rewrite_rules_type,
    create_rewrite_rules, Z
)
from matrix_trajectory_animation import get_matrix_entries_as_coordinates
from trajectory_animation import calculate_trajectory
import ipywidgets as widgets
from IPython.display import display, clear_output

print("✅ Imports successful!")

## 1. Matrix Visualization Functions

In [ ]:
def plot_matrix(matrix, title="Matrix", ax=None, show_coords=True):
    """Plot a boolean matrix with optional coordinate labels."""
    if not matrix:
        print(f"{title}: Empty matrix")
        return
    
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    
    # Convert boolean matrix to numeric for plotting
    plot_matrix = np.array([[1 if cell else 0 for cell in row] for row in matrix])
    
    # Plot with blue for filled, white for empty
    im = ax.imshow(plot_matrix, cmap='Blues', vmin=0, vmax=1)
    
    # Add grid
    ax.set_xticks(np.arange(-0.5, len(matrix[0]), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(matrix), 1), minor=True)
    ax.grid(which='minor', color='gray', linestyle='-', linewidth=1)
    
    # Add coordinate labels if requested
    if show_coords:
        for i in range(len(matrix)):
            for j in range(len(matrix[0])):
                if matrix[i][j]:
                    ax.text(j, i, f'({i},{j})', ha='center', va='center', 
                           fontsize=8, color='white', weight='bold')
    
    ax.set_title(title)
    ax.set_aspect('equal')
    
    return ax

def plot_coordinates(coordinates, title="Coordinates", ax=None, max_size=8):
    """Plot coordinates as a matrix visualization."""
    if not coordinates:
        print(f"{title}: No coordinates")
        return
    
    # Determine matrix size
    max_coord = max(max(coord) for coord in coordinates)
    size = min(max_size, max(2, max_coord + 1))
    
    # Create boolean matrix
    matrix = [[False for _ in range(size)] for _ in range(size)]
    for x, y in coordinates:
        if x < size and y < size:
            matrix[x][y] = True
    
    return plot_matrix(matrix, title, ax)

def compare_matrices(matrix1, matrix2, title1="Matrix 1", title2="Matrix 2"):
    """Side-by-side comparison of two matrices."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    plot_matrix(matrix1, title1, ax1)
    plot_matrix(matrix2, title2, ax2)
    plt.tight_layout()
    plt.show()

print("📊 Visualization functions ready!")

## 2. Configuration Explorer

In [ ]:
# Interactive configuration
curve_dropdown = widgets.Dropdown(
    options=['morton', 'hilbert'],
    value='morton',
    description='Curve Type:'
)

rules_dropdown = widgets.Dropdown(
    options=['default', 'sandwich'],
    value='default',
    description='Rewrite Rules:'
)

number_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=64,
    step=1,
    description='Number:'
)

def update_config(curve_type, rules_type, number):
    """Update configuration and show resulting matrix."""
    set_matrix_curve_type(curve_type)
    set_rewrite_rules_type(rules_type)
    
    # Create matrix graph
    node = create_matrix_graph(number)
    matrix = node.to_matrix()
    coords = get_matrix_entries_as_coordinates(node)
    
    # Display info
    clear_output(wait=True)
    print(f"Configuration: {curve_type.title()} + {rules_type.title()} Rules")
    print(f"Number: {number} (binary: {bin(number)})")
    print(f"Coordinates: {coords}")
    print(f"Matrix entries: {len(node.entries)}")
    
    # Plot matrix
    plot_matrix(matrix, f"{curve_type.title()} Curve - Number {number}")
    plt.show()

# Create interactive widget
interactive_config = widgets.interactive(
    update_config,
    curve_type=curve_dropdown,
    rules_type=rules_dropdown,
    number=number_slider
)

display(interactive_config)
print("🎛️ Interactive configuration ready!")

## 3. Rewrite Rules Comparison

In [ ]:
def compare_rewrite_rules(number, curve_type='hilbert'):
    """Compare default vs sandwich rewrite rules for a given number."""
    set_matrix_curve_type(curve_type)
    
    # Test with default rules
    set_rewrite_rules_type('default')
    default_node = create_matrix_graph(number)
    default_matrix = default_node.to_matrix()
    default_coords = get_matrix_entries_as_coordinates(default_node)
    
    # Test with sandwich rules
    set_rewrite_rules_type('sandwich')
    sandwich_node = create_matrix_graph(number)
    sandwich_matrix = sandwich_node.to_matrix()
    sandwich_coords = get_matrix_entries_as_coordinates(sandwich_node)
    
    print(f"Comparing rewrite rules for number {number} ({curve_type} curve)")
    print(f"Binary: {bin(number)}")
    print(f"Default coordinates: {default_coords}")
    print(f"Sandwich coordinates: {sandwich_coords}")
    print(f"Same result: {default_coords == sandwich_coords}")
    
    # Plot comparison
    compare_matrices(
        default_matrix, sandwich_matrix,
        f"Default Rules - {number}", f"Sandwich Rules - {number}"
    )
    
    return default_coords, sandwich_coords

# Test a few numbers
for n in [5, 15, 21]:
    compare_rewrite_rules(n)
    print("\n" + "="*50 + "\n")

## 4. Trajectory Analysis

In [ ]:
def analyze_trajectory(start_number, curve_type='hilbert', rules_type='default', max_steps=5):
    """Analyze the trajectory of matrix transformations through rewrite rules."""
    set_matrix_curve_type(curve_type)
    set_rewrite_rules_type(rules_type)
    
    # Create starting node
    start_node = create_matrix_graph(start_number)
    
    # Calculate trajectory
    try:
        trajectory = calculate_trajectory(start_node, max_steps=max_steps)
        
        print(f"Trajectory for {start_number} ({curve_type} + {rules_type})")
        print(f"Trajectory length: {len(trajectory)} steps")
        
        # Plot trajectory progression
        if len(trajectory) > 1:
            fig, axes = plt.subplots(1, min(len(trajectory), 5), figsize=(15, 3))
            if len(trajectory) == 1:
                axes = [axes]
            
            for i, node in enumerate(trajectory[:5]):
                matrix = node.to_matrix()
                coords = get_matrix_entries_as_coordinates(node)
                plot_matrix(matrix, f"Step {i}\n{coords}", axes[i] if len(trajectory) > 1 else axes[0], show_coords=False)
            
            plt.tight_layout()
            plt.show()
        else:
            # Single step - just show the matrix
            matrix = trajectory[0].to_matrix()
            coords = get_matrix_entries_as_coordinates(trajectory[0])
            plot_matrix(matrix, f"Start: {coords}")
            plt.show()
        
        return trajectory
        
    except Exception as e:
        print(f"Error calculating trajectory: {e}")
        return [start_node]

# Test trajectory analysis
trajectory1 = analyze_trajectory(5, 'hilbert', 'default')
print("\n" + "="*30 + "\n")
trajectory2 = analyze_trajectory(5, 'hilbert', 'sandwich')

## 5. Custom Rewrite Rule Testing

In [ ]:
def test_rewrite_operations(left_number, right_number, curve_type='hilbert', rules_type='default'):
    """Test rewrite operations between two specific numbers."""
    set_matrix_curve_type(curve_type)
    set_rewrite_rules_type(rules_type)
    
    # Create nodes
    left_node = create_matrix_graph(left_number)
    right_node = create_matrix_graph(right_number)
    
    print(f"Testing {left_number} + {right_number} with {rules_type} rules ({curve_type} curve)")
    
    # Get matrices for visualization
    left_matrix = left_node.to_matrix()
    right_matrix = right_node.to_matrix()
    left_coords = get_matrix_entries_as_coordinates(left_node)
    right_coords = get_matrix_entries_as_coordinates(right_node)
    
    print(f"Left ({left_number}): {left_coords}")
    print(f"Right ({right_number}): {right_coords}")
    
    # Test rewrite operation
    try:
        if hasattr(left_node, 'rewrite_from_input'):
            result_entries = left_node.rewrite_from_input(right_node)
            print(f"Rewrite result: {len(result_entries)} entries")
            
            for i, entry in enumerate(result_entries[:3]):  # Show first 3
                print(f"  Entry {i+1}: {entry}")
        else:
            print("No rewrite_from_input method available")
    
        # Show input matrices
        compare_matrices(left_matrix, right_matrix, 
                        f"Left: {left_number}", f"Right: {right_number}")
    
    except Exception as e:
        print(f"Error in rewrite operation: {e}")

# Test different combinations
test_rewrite_operations(5, 3, 'hilbert', 'default')
print("\n" + "="*40 + "\n")
test_rewrite_operations(5, 3, 'hilbert', 'sandwich')

## 6. Batch Analysis and Comparison

In [ ]:
def batch_analysis(numbers, curve_types=['morton', 'hilbert'], rules_types=['default', 'sandwich']):
    """Analyze multiple numbers across different configurations."""
    results = {}
    
    for curve in curve_types:
        for rules in rules_types:
            config_key = f"{curve}_{rules}"
            results[config_key] = {}
            
            set_matrix_curve_type(curve)
            set_rewrite_rules_type(rules)
            
            for num in numbers:
                node = create_matrix_graph(num)
                coords = get_matrix_entries_as_coordinates(node)
                results[config_key][num] = coords
    
    # Display results table
    print(f"{'Number':<8}", end="")
    for config in results.keys():
        print(f"{config:<20}", end="")
    print()
    print("="*80)
    
    for num in numbers:
        print(f"{num:<8}", end="")
        for config in results.keys():
            coords = results[config][num]
            coord_str = str(coords)[:18] + ".." if len(str(coords)) > 20 else str(coords)
            print(f"{coord_str:<20}", end="")
        print()
    
    return results

# Run batch analysis
test_numbers = [1, 3, 5, 7, 15, 21, 31]
batch_results = batch_analysis(test_numbers)

## 7. Custom Experimentation Sandbox

In [ ]:
# Sandbox cell for custom experiments
print("🧪 Experimentation Sandbox")
print("Use this cell to test custom combinations and explore specific scenarios")
print()

# Example: Custom experiment
def custom_experiment():
    """Template for custom experiments."""
    # Your custom code here
    set_matrix_curve_type('hilbert')
    set_rewrite_rules_type('sandwich')
    
    # Example: Look for interesting patterns
    interesting_numbers = []
    for n in range(1, 32):
        node = create_matrix_graph(n)
        coords = get_matrix_entries_as_coordinates(node)
        
        # Custom criteria - e.g., numbers with exactly 3 coordinates
        if len(coords) == 3:
            interesting_numbers.append((n, coords))
    
    print("Numbers with exactly 3 coordinates:")
    for num, coords in interesting_numbers:
        print(f"  {num}: {coords}")
    
    return interesting_numbers

# Run custom experiment
custom_results = custom_experiment()

## 8. Export and Save Results

In [ ]:
import json
from datetime import datetime

def save_experiment_results(results, filename=None):
    """Save experiment results to JSON file."""
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"rewrite_experiment_{timestamp}.json"
    
    # Convert results to JSON-serializable format
    json_results = {
        'timestamp': datetime.now().isoformat(),
        'results': results
    }
    
    with open(filename, 'w') as f:
        json.dump(json_results, f, indent=2)
    
    print(f"✅ Results saved to {filename}")
    return filename

# Save the batch results if you want
# save_experiment_results(batch_results)

print("💾 Export functions ready!")
print("\n🎉 Rewrite Rules Explorer is ready for use!")
print("\nTips:")
print("- Use the interactive widgets in section 2 to explore configurations")
print("- Modify section 7 for custom experiments")
print("- Use compare_rewrite_rules() to see differences between rule types")
print("- analyze_trajectory() shows how matrices evolve through rewrites")